<a href="https://colab.research.google.com/github/karthiknagar16/mean-course/blob/master/nb/Phi_4_(14B)-GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** — a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

Load up `Phi-4 14B`, and set parameters

In [3]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from unsloth.chat_templates import get_chat_template

# Total sequence length (e.g., 2048 or 4096)
max_seq_length = 2048  # 512 # Can increase for longer reasoning traces
lora_rank = 16 #16 # Larger rank = smarter, but slower

#model_name = "unsloth/Phi-4"
model_name = "HuggingFaceTB/SmolLM2-135M"
# model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

#For base model we need to add chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",   # for SmolLM2-style chat data
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["gate_proj", "up_proj", "down_proj",],
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 04-04 05:30:32 [__init__.py:244] Automatically detected platform cuda.
ERROR 04-04 05:30:37 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-04 05:30:49 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
INFO 04-04 05:30:49 [vllm_utils.py:753] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2026.4.2: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standb

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 04-04 05:31:14 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 04-04 05:31:14 [config.py:1472] Using max model len 2048
WARNING 04-04 05:31:14 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 04-04 05:31:15 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=4096.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'fp4', 'bnb_4bit_use_double_quant': False, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': [], 'llm_int8_threshold': 6.0}
INFO 04-04 05:31:15 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='HuggingFaceTB/SmolLM2-135M', speculative_config=None, tokenizer='HuggingFaceTB/SmolLM2-135M', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neur

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

HuggingFaceTB/SmolLM2-135M does not have a padding token! Will use pad_token = <|endoftext|>.


Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.


Model does not have a padding token! Will use pad_token = <|endoftext|>.


Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.2 patched 30 layers with 0 QKV layers, 0 O layers and 30 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [ ]:
# old reward functions - which generated zero rewards all time
import re
from datasets import load_dataset, Dataset
from pathlib import Path
import pandas as pd

# Load and prep dataset

# Existing system prompt :
# SYSTEM_PROMPT = """
# Respond in the following format:
# <reasoning>
# ...
# </reasoning>
# <answer>
# ...
# </answer>
# """

# def extract_xml_answer(text: str) -> str:
#     answer = text.split("<answer>")[-1]
#     answer = answer.split("</answer>")[0]
#     return answer.strip()

# New system prompt for competation :
SYSTEM_PROMPT = r"""
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
\boxed{...}
</answer>

Always put the final answer inside a single \boxed{} LaTeX command.
Do not put anything outside the XML tags.
"""

def extract_xml_answer(text: str) -> str | None:
    answer = text.split("<answer>")[-1].split("</answer>")[0].strip()
    match = re.search(r"\\boxed\{(.*)\}", answer, re.DOTALL)
    return match.group(1).strip() if match else answer


XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
# def get_gsm8k_questions(split = "train") -> Dataset:
#     data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
#     data = data.map(lambda x: { # type: ignore
#         'prompt': [
#             {'role': 'system', 'content': SYSTEM_PROMPT},
#             {'role': 'user', 'content': x['question']}
#         ],
#         'answer': extract_hash_answer(x['answer'])
#     }) # type: ignore
#     return data # type: ignore

# dataset = get_gsm8k_questions()

TRAIN_CSV = Path("train.csv")
NUM_SAMPLES = 3000
def get_train_csv_questions(csv_path: str | Path = TRAIN_CSV) -> Dataset:
    df = pd.read_csv(csv_path).head(NUM_SAMPLES)

    required_columns = {"prompt", "answer"}
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {csv_path}: {missing}")

    data = Dataset.from_pandas(df, preserve_index=False)

    data = data.map(lambda x: {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(x["prompt"])}
        ],
        "answer": str(x["answer"]).strip()
    })

    return data

dataset = get_train_csv_questions()

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

# def int_reward_func(completions, **kwargs) -> list[float]:
#     responses = [completion[0]['content'] for completion in completions]
#     extracted_responses = [extract_xml_answer(r) for r in responses]
#     return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
dataset[0]

{'id': '00066667',
 'prompt': [{'content': '\nRespond in the following format:\n<reasoning>\n...\n</reasoning>\n<answer>\n\\boxed{...}\n</answer>\n\nAlways put the final answer inside a single \\boxed{} LaTeX command.\nDo not put anything outside the XML tags.\n',
   'role': 'system'},
  {'content': "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01010001 -> 11011101\n00001001 -> 01101101\n00010101 -> 01010101\n11111111 -> 10000001\n10011101 -> 01000101\n00111011 -> 00001001\n10111101 -> 00000101\n00100110 -> 10110011\n\nNow, determine the output for: 00110100",
   'role': 'user'}],
 'answer': '10010111'}

In [4]:
# New Reward functions :

import re
from datasets import load_dataset, Dataset
from pathlib import Path
import pandas as pd

# ================================
# System Prompt
# ================================

SYSTEM_PROMPT = r"""
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
\boxed{...}
</answer>

Always put the final answer inside a single \boxed{} LaTeX command.
Do not put anything outside the XML tags.
"""


# ================================
# Extract Functions
# ================================

def extract_xml_answer(text: str) -> str | None:
    try:
        answer = text.split("<answer>")[-1].split("</answer>")[0].strip()
        match = re.search(r"\\boxed\{(.*?)\}", answer, re.DOTALL)  # non-greedy
        return match.group(1).strip() if match else None
    except:
        return None


XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""


def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()


# ================================
# Dataset Loading
# ================================

TRAIN_CSV = Path("train.csv")
NUM_SAMPLES = 3000


def get_train_csv_questions(csv_path: str | Path = TRAIN_CSV) -> Dataset:
    df = pd.read_csv(csv_path).head(NUM_SAMPLES)

    required_columns = {"prompt", "answer"}
    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns in {csv_path}: {missing}")

    data = Dataset.from_pandas(df, preserve_index=False)

    data = data.map(lambda x: {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(x["prompt"])}
        ],
        "answer": str(x["answer"]).strip()
    })

    return data


dataset = get_train_csv_questions()


# ================================
# Reward Functions
# ================================

# ------------------------------------------------
# Base reward (prevents zero reward collapse)
# ------------------------------------------------

def base_reward_func(completions, **kwargs) -> list[float]:
    return [0.05 for _ in completions]


# ------------------------------------------------
# Length reward (encourage reasoning)
# ------------------------------------------------

def length_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]["content"] for completion in completions]
    rewards = [min(len(r) / 200, 0.3) for r in responses]
    return rewards


# ------------------------------------------------
# Boxed reward (bootstrap learning)
# ------------------------------------------------

def boxed_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]["content"] for completion in completions]

    rewards = []
    for r in responses:
        if "\\boxed{" in r:
            rewards.append(0.3)
        else:
            rewards.append(0.0)

    return rewards


# ------------------------------------------------
# Correctness reward (graded reward)
# ------------------------------------------------

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:

    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]

    rewards = []

    for r, a in zip(extracted_responses, answer):

        if r == a:
            rewards.append(2.0)

        elif r is not None:
            rewards.append(0.2)

        else:
            rewards.append(0.0)

    return rewards


# ------------------------------------------------
# Integer reward
# ------------------------------------------------

# def int_reward_func(completions, **kwargs) -> list[float]:

#     responses = [completion[0]['content'] for completion in completions]
#     extracted_responses = [extract_xml_answer(r) for r in responses]

#     rewards = []

#     for r in extracted_responses:

#         if r is not None and r.isdigit():
#             rewards.append(0.5)
#         else:
#             rewards.append(0.0)

#     return rewards


# ------------------------------------------------
# Strict Format Reward
# ------------------------------------------------

def strict_format_reward_func(completions, **kwargs) -> list[float]:

    pattern = r"^<reasoning>.*?</reasoning>.*?<answer>.*?</answer>$"

    responses = [completion[0]["content"] for completion in completions]

    matches = [re.search(pattern, r, re.DOTALL) for r in responses]

    return [0.5 if match else 0.0 for match in matches]


# ------------------------------------------------
# Soft Format Reward
# ------------------------------------------------

def soft_format_reward_func(completions, **kwargs) -> list[float]:

    pattern = r"<reasoning>.*?</reasoning>.*?<answer>.*?</answer>"

    responses = [completion[0]["content"] for completion in completions]

    matches = [re.search(pattern, r, re.DOTALL) for r in responses]

    return [0.5 if match else 0.0 for match in matches]


# ------------------------------------------------
# XML count reward
# ------------------------------------------------

def count_xml(text) -> float:

    count = 0.0

    if text.count("<reasoning>") == 1:
        count += 0.125

    if text.count("</reasoning>") == 1:
        count += 0.125

    if text.count("<answer>") == 1:
        count += 0.125

    if text.count("</answer>") == 1:
        count += 0.125

    return count


def xmlcount_reward_func(completions, **kwargs) -> list[float]:

    contents = [completion[0]["content"] for completion in completions]

    return [count_xml(c) for c in contents]


# ================================
# Final Reward List
# ================================

reward_funcs = [

    base_reward_func,          # prevents zero reward
    length_reward_func,        # encourage reasoning
    boxed_reward_func,         # bootstrap learning
    xmlcount_reward_func,      # XML format
    soft_format_reward_func,   # soft formatting
    #int_reward_func,           # integer reward
    correctness_reward_func    # correctness (main reward)

]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [5]:
dataset[0]

{'id': '00066667',
 'prompt': [{'content': '\nRespond in the following format:\n<reasoning>\n...\n</reasoning>\n<answer>\n\\boxed{...}\n</answer>\n\nAlways put the final answer inside a single \\boxed{} LaTeX command.\nDo not put anything outside the XML tags.\n',
   'role': 'system'},
  {'content': "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01010001 -> 11011101\n00001001 -> 01101101\n00010101 -> 01010101\n11111111 -> 10000001\n10011101 -> 01000101\n00111011 -> 00001001\n10111101 -> 00000101\n00100110 -> 10110011\n\nNow, determine the output for: 00110100",
   'role': 'user'}],
 'answer': '10010111'}

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [9]:
from trl import GRPOConfig, GRPOTrainer



# Set prompt length to fit your longest question
max_prompt_length = 512

training_args = GRPOConfig(
    temperature = 1.0,           # Recommended starting point for reasoning
    num_generations = 8,         # 'G' in GRPO; higher is better for stats but uses VRAM , #6 # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = (max_seq_length - max_prompt_length),  # Give it enough room to "think" # 200
    use_vllm = True, # use vllm for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    logging_first_step=True, # extra added
    per_device_train_batch_size = 16, # 64, #48, #16 #1
    gradient_accumulation_steps = 4, # 64, # 48 # 16 #1 # Increase to 4 for smoother training
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 300,
    save_steps = 50, #100, #250
    eval_steps = 5, # extra added
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [10]:
# TEST
print(tokenizer.chat_template is not None)
print(tokenizer.apply_chat_template(dataset[0]["prompt"], tokenize=False, add_generation_prompt=True))

True
<|im_start|>system

Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
\boxed{...}
</answer>

Always put the final answer inside a single \boxed{} LaTeX command.
Do not put anything outside the XML tags.
<|im_end|>
<|im_start|>user
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100<|im_end|>
<|im_start|>assistant



In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    # reward_funcs = [
    #     xmlcount_reward_func,
    #     soft_format_reward_func,
    #     strict_format_reward_func,
    #     # int_reward_func,
    #     correctness_reward_func,
    # ],
    reward_funcs = reward_funcs,
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 3,041,280 of 137,556,288 (2.21% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / base_reward_func / mean,rewards / base_reward_func / std,rewards / length_reward_func / mean,rewards / length_reward_func / std,rewards / boxed_reward_func / mean,rewards / boxed_reward_func / std,rewards / xmlcount_reward_func / mean,rewards / xmlcount_reward_func / std,rewards / soft_format_reward_func / mean,rewards / soft_format_reward_func / std,rewards / int_reward_func / mean,rewards / int_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,-0.125500,0.493750,0.273788,345.437500,8.000000,1536.000000,0.062500,266.066681,8.000000,1282.000000,0.000000,0.050000,0.000000,0.287500,0.043404,0.075000,0.131982,0.000000,0.000000,0.000000,0.000000,0.031250,0.122967,0.050000,0.087988
2,0.074100,0.584375,0.278835,312.312500,28.000000,1536.000000,0.062500,230.733353,28.000000,1415.000000,0.000000,0.050000,0.000000,0.300000,0.000000,0.121875,0.149697,0.000000,0.000000,0.000000,0.000000,0.031250,0.122967,0.081250,0.099798
3,0.049800,0.525469,0.239954,351.312500,8.000000,1536.000000,0.093750,228.758621,8.000000,795.000000,0.000029,0.050000,0.000000,0.291875,0.045962,0.103125,0.144768,0.011719,0.048769,0.000000,0.000000,0.000000,0.000000,0.068750,0.096512
4,-0.136100,0.656250,0.311406,805.718750,55.000000,1536.000000,0.406250,306.052643,55.000000,940.000000,0.000027,0.050000,0.000000,0.300000,0.000000,0.168750,0.151205,0.015625,0.088388,0.015625,0.088388,0.000000,0.000000,0.106250,0.101401
5,-0.285400,0.592188,0.291781,706.375000,20.000000,1536.000000,0.375000,208.600006,20.000000,721.000000,0.000028,0.050000,0.000000,0.300000,0.000000,0.131250,0.151205,0.007812,0.030742,0.000000,0.000000,0.015625,0.088388,0.087500,0.100803
6,-0.019300,0.552500,0.275942,711.656250,7.000000,1536.000000,0.343750,279.857147,7.000000,1142.000000,0.000028,0.050000,0.000000,0.291562,0.047730,0.093750,0.141279,0.007812,0.030742,0.000000,0.000000,0.046875,0.148072,0.062500,0.094186
7,-0.069900,0.572656,0.254162,660.593750,95.000000,1536.000000,0.250000,368.791687,95.000000,1462.000000,0.000029,0.050000,0.000000,0.300000,0.000000,0.131250,0.151205,0.003906,0.022097,0.000000,0.000000,0.000000,0.000000,0.087500,0.100803
8,-0.393200,0.606250,0.287769,280.875000,11.000000,1536.000000,0.062500,197.200012,11.000000,962.000000,0.000029,0.050000,0.000000,0.290625,0.034024,0.140625,0.152102,0.000000,0.000000,0.000000,0.000000,0.031250,0.122967,0.093750,0.101401
9,0.095300,0.515625,0.268371,560.218750,17.000000,1536.000000,0.250000,234.958344,17.000000,817.000000,0.000026,0.050000,0.000000,0.300000,0.000000,0.093750,0.141279,0.000000,0.000000,0.000000,0.000000,0.015625,0.088388,0.056250,0.091361
10,-0.206200,0.597344,0.324277,380.375000,7.000000,1536.000000,0.156250,166.370377,7.000000,569.000000,0.000029,0.050000,0.000000,0.276250,0.064220,0.131250,0.151205,0.011719,0.048769,0.000000,0.000000,0.046875,0.148072,0.081250,0.099798


<a name="Inference"></a>
### Inference

Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Which is bigger? 9.11 or 9.9?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"To determine which one is bigger, we need to compare their sizes. Let's count the number of items in each case:\n\n9.11: There are 9 items.\n9.9: There are 9.9 items.\n\nAs you can see, both 9.9 and 9.11 are the same size. Therefore, 9.9 is the largest denomination in both cases.\n\nSo, the largest denomination in both cases is 9.9."

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Which is bigger? 9.11 or 9.9?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"I'm not sure which one is bigger, but I can say that 9.9 is larger than 9.11.\n\nNote that I'm not the only one here, but I have this bit of information:\n\nI'm not sure which one is bigger.\n\nI'm not sure which one is bigger, but I can say that 9.9 is larger than 9.11.\n\nBoth 9.9 and 9.11 are reported to be larger than 9.9."

In [ ]:
print(output)

8.9 is smaller than 9.11, but 9.11 is smaller than 9.9.


Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("phi_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/phi_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("phi_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/phi_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("phi_lora")
    tokenizer.save_pretrained("phi_lora")
if False:
    model.push_to_hub("HF_USERNAME/phi_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/phi_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("phi_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/phi_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("phi_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/phi_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("phi_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/phi_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/phi_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Now, use the `phi_finetune.Q8_0.gguf` file or `phi_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).